# 6장. Gradio 기반 LLM 챗봇 만들기

이 장은 `LLM/llm2.ipynb`의 Gradio 챗봇 예제들과 PDF 교재의 LangChain, Ollama, RAG 활용 사례를 통합한 장입니다.

## 이 장에서 다루는 내용

1. Gradio란 무엇인가
2. `Interface`, `ChatInterface`, `Blocks`
3. 역할 부여 챗봇
4. Gradio 기본 챗봇
5. ChatOllama와 대화 이력
6. CSV 기반 RAG 챗봇
7. URL 기반 RAG 챗봇
8. Gradio Tab 구성
9. 서버 실행과 종료
10. 포트 충돌과 `share`, `debug` 옵션

이 장의 핵심은 노트북 코드로 만든 LLM 기능을 사용자가 입력하고 결과를 볼 수 있는 웹 UI로 만드는 것입니다.


## 6.1 Gradio란?

Gradio는 Python 함수에 웹 UI를 빠르게 붙일 수 있는 라이브러리입니다.

LLM 실습에서 Gradio가 유용한 이유는 다음과 같습니다.

- 웹 프론트엔드를 직접 만들지 않아도 됩니다.
- 텍스트, 이미지, 오디오, 파일 업로드 UI를 쉽게 만들 수 있습니다.
- 챗봇 인터페이스를 간단히 만들 수 있습니다.
- Jupyter Notebook에서 바로 실행해 브라우저로 확인할 수 있습니다.
- LLM, RAG, STT/TTS, 이미지 분석 실습을 빠르게 데모로 만들 수 있습니다.

원본 `llm2.ipynb`에서는 Gradio를 다음 실습에 사용합니다.

- 일반 챗봇
- CSV 기반 대학 정보 챗봇
- URL 기반 RAG 챗봇
- STT/TTS 앱
- 이미지 분류 앱
- DB 질의 챗봇
- 자기소개서 도우미 챗봇


## 6.2 Gradio 주요 구성 방식

원본 노트북에는 세 가지 Gradio 구성 방식이 등장합니다.

| 방식 | 설명 | 사용 예 |
|---|---|---|
| `gr.Interface` | 하나의 함수에 입력/출력 컴포넌트를 연결하는 가장 간단한 방식 | STT, TTS, 이미지 분류 |
| `gr.ChatInterface` | 챗봇 UI를 빠르게 만드는 방식 | 기본 LLM 챗봇, CSV RAG 챗봇 |
| `gr.Blocks` | Row, Column, Tab 등 복잡한 레이아웃을 직접 구성하는 방식 | URL RAG 탭 UI, DB 챗봇, 자기소개서 도우미 |

이 장에서는 챗봇 중심으로 `ChatInterface`와 `Blocks`를 먼저 다룹니다.


## 6.3 설치와 기본 import

원본 노트북의 Gradio 관련 설치 주석은 다음과 같습니다.

```python
pip install gradio
```

의존성 문제가 생기면 다음처럼 처리할 수 있습니다.

```python
pip install --upgrade pydantic
```

또는 문제가 심한 경우 기존 pydantic을 제거한 뒤 재설치할 수 있습니다.

```python
pip uninstall pydantic pydantic-core -y
pip install gradio
```


In [ ]:
# Gradio와 LangChain 챗봇 실습용 설치 예시입니다.
# 필요한 경우 주석을 해제하고 실행하세요.

# %pip install gradio
# %pip install langchain langchain-community langchain-core


In [ ]:
# Gradio는 웹 UI를 만들기 위해 사용합니다.
import gradio as gr

# ChatOllama는 Ollama 모델을 대화형 LangChain 모델로 연결합니다.
from langchain_community.chat_models import ChatOllama

# HumanMessage와 AIMessage는 대화 이력을 LangChain 메시지 형식으로 표현합니다.
from langchain_core.messages import HumanMessage, AIMessage


## 6.4 역할 부여 챗봇 복습

`llm2.ipynb`의 첫 번째 챗봇 예제는 Gradio 없이 Python 함수로 Ollama를 직접 호출합니다.

핵심은 system prompt입니다.

```text
너는 전문 심리 상담가야. 질문에 대한 답은 3줄 이내로 짧게해줘.
```

이런 역할 부여는 챗봇을 만들 때 매우 중요합니다.

- 상담가 챗봇
- 교사 챗봇
- 고객센터 상담원
- 문서 요약 전문가
- SQL 생성 전문가

모두 system prompt로 역할과 답변 규칙을 지정할 수 있습니다.


In [ ]:
# Python ollama 패키지로 직접 호출하는 역할 부여 챗봇 예시입니다.
import ollama

def ask_gemma(question):
    # 챗봇의 역할과 답변 길이 규칙을 정의합니다.
    chatbot_role = "너는 전문 심리 상담가야. 질문에 대한 답은 3줄 이내로 짧게해줘."

    # system 메시지와 user 메시지를 함께 보냅니다.
    response = ollama.chat(
        model='gemma4:e2b',
        messages=[
            {"role": "system", "content": chatbot_role},
            {"role": "user", "content": question},
        ]
    )

    # 모델 응답 텍스트를 반환합니다.
    return response['message']['content']


In [ ]:
# 역할 부여 챗봇 테스트 예시입니다.
question = "행복하기 위해 어떻게 하면 좋을까?"
response = ask_gemma(question)
print(response)


## 6.5 Gradio ChatInterface 기본 챗봇

`gr.ChatInterface`는 챗봇 UI를 가장 빠르게 만들 수 있는 방식입니다.

필요한 것은 다음 형태의 함수입니다.

```python
def chat(message, history):
    ...
    return response
```

인자 의미:

| 인자 | 의미 |
|---|---|
| `message` | 사용자가 방금 입력한 메시지 |
| `history` | 이전 대화 기록 |

`history`는 이전 사용자 입력과 AI 응답의 쌍으로 전달됩니다. 이 값을 모델에 함께 넣으면 대화 문맥을 이어갈 수 있습니다.


In [ ]:
# ChatOllama 모델을 초기화합니다.
# temperature가 낮으면 더 안정적이고, 높으면 더 다양하고 창의적인 답변이 나올 수 있습니다.
model = ChatOllama(model="gemma4:e2b", temperature=0.7, verbose=False)


In [ ]:
# 채팅 기록을 포함하여 응답을 생성하는 함수입니다.
def chat(message, history):
    # 이전 대화 기록을 ChatOllama가 이해하는 메시지 형식으로 변환합니다.
    chat_history = []
    for human, ai in history:
        chat_history.append(HumanMessage(content=human))
        chat_history.append(AIMessage(content=ai))

    # 현재 사용자 메시지를 추가합니다.
    chat_history.append(HumanMessage(content=message))

    # 모델을 호출합니다.
    response = model.invoke(chat_history)

    # 응답 텍스트만 반환합니다.
    return response.content


In [ ]:
# Gradio ChatInterface를 설정합니다.
demo = gr.ChatInterface(
    fn=chat,
    examples=[
        "안녕하세요!",
        "인공지능에 대해 설명해주세요.",
        "파이썬의 장점은 무엇인가요?"
    ],
    title="AI 챗봇",
    description="질문을 입력하면 AI가 답변합니다."
)


In [ ]:
# Gradio 서버를 실행합니다.
# 같은 포트를 이미 사용 중이면 7862, 7863 등으로 바꾸세요.
demo.launch(server_port=7861, server_name="0.0.0.0")


In [ ]:
# Gradio 서버를 종료합니다.
demo.close()


## 6.6 CSV 기반 RAG 챗봇 개요

`llm2.ipynb`에는 CSV 파일을 기반으로 질문에 답하는 챗봇 예제가 있습니다.

사용 데이터:

```text
./dataset/indata_kor.csv
```

인코딩:

```python
encoding='CP949'
```

처리 흐름:

1. CSV 파일을 Pandas DataFrame으로 읽습니다.
2. DataFrame 내용을 문자열로 변환합니다.
3. 텍스트를 청크로 나눕니다.
4. Hugging Face 임베딩 모델로 벡터화합니다.
5. FAISS 벡터스토어를 만듭니다.
6. Retriever로 관련 청크를 검색합니다.
7. 검색 context와 질문을 프롬프트에 넣습니다.
8. ChatOllama가 답변합니다.
9. Gradio ChatInterface로 사용자에게 제공합니다.

이 예제는 PDF 6.4의 문서 기반 질의응답 활용 사례와 연결됩니다.


In [ ]:
# CSV RAG 챗봇에 필요한 모듈입니다.
import pandas as pd
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough


In [ ]:
# CSV 파일을 로드합니다.
# Windows 한국어 CSV 파일은 CP949 인코딩인 경우가 많습니다.
df = pd.read_csv("./dataset/indata_kor.csv", encoding='CP949')

# 마지막 일부 행을 확인합니다.
df.tail()


In [ ]:
# CSV 내용을 하나의 긴 문자열로 만든 뒤 청크로 나눕니다.
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_text("\n".join(df.to_string()))

# 임베딩 모델을 초기화합니다.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# FAISS 벡터스토어를 생성합니다.
vectorstore = FAISS.from_texts(texts, embeddings)

# 검색기를 생성합니다. k=1은 가장 관련 있는 청크 하나만 가져온다는 의미입니다.
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})


In [ ]:
# CSV RAG 답변 생성을 위한 ChatOllama 모델입니다.
# 원본 노트북에는 tempeature 오타가 있으므로 여기서는 temperature로 표기합니다.
llm = ChatOllama(model="gemma2", temperature=0.1)

# 대화 이력과 context를 포함하는 프롬프트 템플릿입니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer based on the provided context."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}\n\nContext: {context}")
])


In [ ]:
# 검색된 문서 목록을 LLM이 읽기 좋은 문자열로 바꿉니다.
def format_docs(docs):
    # 검색 결과가 없을 때 사용할 기본 문구입니다.
    if not docs:
        return "No context available"

    # 문서 내용을 모을 리스트입니다.
    result = []

    # 각 문서의 page_content를 추출합니다.
    for doc in docs:
        if hasattr(doc, 'page_content'):
            result.append(doc.page_content)

    # 여러 문서를 줄바꿈으로 연결합니다.
    return "\n".join(result)


In [ ]:
# CSV 기반 RAG 챗봇 함수입니다.
def csv_rag_chat(message, history):
    # 이전 대화 기록을 LangChain 메시지로 변환합니다.
    chat_history = []
    for human, ai in history:
        chat_history.append(HumanMessage(content=human))
        chat_history.append(AIMessage(content=ai))

    # 사용자 질문으로 관련 문서를 검색합니다.
    docs = retriever.invoke(message)
    context = format_docs(docs)

    # 프롬프트 메시지를 구성합니다.
    messages = prompt.format_messages(
        chat_history=chat_history,
        question=message,
        context=context
    )

    # LLM 응답을 생성합니다.
    response = llm.invoke(messages)

    # 문서 metadata에서 source 정보를 추출합니다.
    sources = set([doc.metadata.get('source', 'Unknown') for doc in docs])
    source_info = f"\n\n참고 출처: {', '.join(sources)}" if sources else ""

    # 답변과 출처를 함께 반환합니다.
    return response.content + source_info


In [ ]:
# CSV RAG 챗봇 Gradio 인터페이스입니다.
csv_demo = gr.ChatInterface(
    fn=csv_rag_chat,
    examples=[
        "한국폴리텍대학 스마트금융과 입학 전까지 어떤걸 공부하면 될까요?",
        "스마트금융과에 대해 설명해주세요",
        "한국폴리텍대한 추천할만한 학과 하나를 소개해주세요."
    ],
    title="대학 정보 AI 챗봇",
    description="스마트금융과에 대한 질문을 입력하면 AI가 CSV 데이터를 참고하여 한글로 답변합니다."
)


In [ ]:
# CSV RAG 챗봇 서버를 실행합니다.
csv_demo.launch(server_port=7861, server_name="0.0.0.0")


In [ ]:
# CSV RAG 챗봇 서버를 종료합니다.
csv_demo.close()


## 6.7 URL 기반 RAG 챗봇 개요

`llm2.ipynb`에는 URL을 입력받아 웹페이지 내용을 읽고, 질문에 답하는 RAG 예제가 있습니다.

이 예제는 PDF 6.4의 `실시간 정보 제공` 활용 사례와 연결됩니다.

처리 흐름:

1. 사용자가 URL과 질문을 입력합니다.
2. `WebBaseLoader`가 웹페이지 내용을 로드합니다.
3. `RecursiveCharacterTextSplitter`가 문서를 청크로 나눕니다.
4. `OllamaEmbeddings`가 청크를 벡터로 변환합니다.
5. FAISS 벡터스토어를 만듭니다.
6. Retriever가 질문과 관련된 문서를 찾습니다.
7. Prompt, Ollama, StrOutputParser로 답변을 생성합니다.
8. Gradio Blocks와 Tab으로 UI를 구성합니다.


In [ ]:
# URL RAG에 필요한 모듈입니다.
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.llms import Ollama
from langchain_core.output_parsers import StrOutputParser


In [ ]:
# URL 문서를 로드하고 검색기를 만드는 함수입니다.
def load_and_retrieve_docs(url):
    # 웹페이지 로더를 생성합니다.
    loader = WebBaseLoader(
        web_paths=(url,),
        bs_kwargs=dict()
    )

    # 웹페이지 내용을 문서 객체로 로드합니다.
    docs = loader.load()

    # 문서를 청크로 분할합니다.
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = text_splitter.split_documents(docs)

    # Ollama 임베딩 모델을 사용합니다.
    embeddings = OllamaEmbeddings(model="gemma2")

    # FAISS 벡터스토어를 생성합니다.
    vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)

    # Retriever로 변환해 반환합니다.
    return vectorstore.as_retriever()


In [ ]:
# 검색 문서를 하나의 문자열 context로 합칩니다.
def format_web_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [ ]:
# URL 기반 RAG 체인 함수입니다.
def rag_chain(url, question):
    # URL 문서를 로드하고 검색기를 만듭니다.
    retriever = load_and_retrieve_docs(url)

    # 질문과 관련 있는 문서를 검색합니다.
    retrieved_docs = retriever.invoke(question)
    formatted_context = format_web_docs(retrieved_docs)

    # 질문과 context를 받는 프롬프트를 구성합니다.
    prompt = ChatPromptTemplate.from_template(
        "Question: {question}\n\nContext: {context}"
    )

    # 답변 생성을 담당할 Ollama LLM입니다.
    llm = Ollama(model='gemma3:4b')

    # LCEL 체인을 구성합니다.
    chain = prompt | llm | StrOutputParser()

    # 체인을 실행합니다.
    response = chain.invoke({
        "question": question,
        "context": formatted_context
    })

    # 생성된 답변을 반환합니다.
    return response


## 6.8 Gradio Blocks와 Tab UI

`gr.Blocks`는 복잡한 UI를 만들 때 사용합니다.

원본 URL RAG 예제는 두 개의 탭을 만듭니다.

| 탭 | 내용 |
|---|---|
| 질문과 답변 | URL과 질문을 입력해 RAG 답변을 받습니다. |
| 시각화(워드클라우드) | 향후 워드클라우드 기능을 넣을 공간입니다. |

이 탭 구조는 하나의 앱 안에 여러 기능을 넣을 때 유용합니다.


In [ ]:
# Gradio Tabbed Interface입니다.
with gr.Blocks() as url_iface:
    # 첫 번째 탭: URL 기반 질문 답변
    with gr.Tab("질문과 답변"):
        gr.Interface(
            fn=rag_chain,
            inputs=["text", "text"],
            outputs="text",
            title="RAG Chain Question Answering",
            description="URL과 질문을 입력하면 RAG 체인에서 답변을 받을 수 있습니다."
        )

    # 두 번째 탭: 시각화 공간
    with gr.Tab("시각화 (워드클라우드)"):
        gr.Markdown("이 탭은 시각화를 위한 공간입니다. 워드클라우드 기능이 여기에 추가될 예정입니다.")


In [ ]:
# URL RAG Gradio 앱을 실행합니다.
url_iface.launch(server_port=7861, server_name="0.0.0.0", debug=True)


In [ ]:
# URL RAG Gradio 앱을 종료합니다.
url_iface.close()


## 6.9 Gradio 실행 옵션

원본 노트북에 자주 등장하는 실행 옵션은 다음과 같습니다.

| 옵션 | 설명 |
|---|---|
| `server_port=7861` | Gradio 앱이 사용할 포트입니다. |
| `server_name="0.0.0.0"` | 외부 접속을 허용할 수 있도록 모든 네트워크 인터페이스에서 실행합니다. |
| `debug=True` | 오류 디버깅 정보를 더 자세히 보여줍니다. |
| `share=True` | Gradio가 임시 외부 공유 링크를 생성합니다. |

주의:

- 같은 포트에 여러 앱을 동시에 띄울 수 없습니다.
- `share=True`는 외부에서 접근 가능한 링크를 만들 수 있으므로 민감한 데이터 실습에서는 주의합니다.
- Jupyter에서 Gradio 앱을 실행한 뒤 종료하지 않으면 다음 앱 실행 시 포트 충돌이 날 수 있습니다.


## 6.10 Gradio 챗봇 설계 팁

LLM 챗봇 UI를 만들 때는 다음을 고려합니다.

### 1. 예시 질문 제공

`examples`에 자주 묻는 질문을 넣으면 사용자가 바로 테스트할 수 있습니다.

### 2. title과 description 작성

앱의 목적을 명확하게 보여줍니다.

### 3. 대화 이력 포함 여부

일반 Q/A는 이력이 필요 없을 수 있지만, 챗봇은 이전 대화 문맥이 중요합니다.

### 4. RAG 챗봇은 context 제한

검색된 문서만 기반으로 답하도록 프롬프트에 명시해야 환각을 줄일 수 있습니다.

### 5. 오류 메시지 처리

파일 없음, Ollama 연결 실패, URL 로드 실패, 임베딩 모델 다운로드 실패 등을 사용자에게 알기 쉽게 보여주는 것이 좋습니다.


## 6.11 자주 발생하는 오류와 해결 방법

### 1. 포트 충돌

오류 원인:

- 이전 Gradio 앱이 아직 실행 중입니다.

해결:

- `demo.close()` 또는 `iface.close()` 실행
- 커널 재시작
- 포트를 `7862` 등으로 변경

### 2. Ollama 연결 실패

해결:

- Ollama 앱 실행
- `ollama list` 확인
- 모델 설치 확인

### 3. CSV 인코딩 오류

해결:

- `encoding='CP949'`
- `encoding='utf-8'`
- `encoding='utf-8-sig'`

순서대로 시도합니다.

### 4. URL 로드 실패

가능한 원인:

- 인터넷 연결 문제
- 사이트가 크롤링을 차단
- 로그인 필요한 페이지
- JavaScript 렌더링이 필요한 페이지

### 5. 임베딩 모델 다운로드 지연

처음 실행 시 모델 다운로드가 필요합니다. 네트워크 상태에 따라 오래 걸릴 수 있습니다.


## 6.12 이 장의 핵심 정리

이 장에서 배운 내용을 정리하면 다음과 같습니다.

- Gradio는 Python 함수에 웹 UI를 쉽게 붙이는 도구입니다.
- `ChatInterface`는 챗봇 UI를 빠르게 만들 수 있습니다.
- `Blocks`는 탭, 행, 열이 있는 복잡한 UI를 만들 때 사용합니다.
- system prompt로 챗봇 역할을 부여할 수 있습니다.
- `ChatOllama`와 `HumanMessage`, `AIMessage`를 사용하면 대화 이력을 포함할 수 있습니다.
- CSV 기반 RAG 챗봇은 파일 데이터를 검색해 답변합니다.
- URL 기반 RAG 챗봇은 웹페이지를 로드해 질문에 답합니다.
- Gradio 앱 실행 후에는 `close()`로 종료하는 것이 좋습니다.
- 포트 충돌, Ollama 연결 실패, CSV 인코딩 오류, URL 로드 실패를 자주 점검해야 합니다.

다음 장에서는 RAG 검색증강생성의 기본 개념과 전체 프로세스를 PDF 내용 중심으로 자세히 정리합니다.
